# Two Speeds of Photometry Availability

This notebook uses frozen raw SkyPortal photometry and GCN circulars. The first cell prepares the matched measurement table used by the five-section narrative.

In [1]:
from pathlib import Path
import json, re
import pandas as pd
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
DETAIL_DIR = ROOT / "data" / "raw" / "skyportal" / "source_detail_20260724"
ARCHIVE_DIR = ROOT / "data" / "raw" / "gcn" / "circulars" / "archive_json" / "20260720_093324" / "extracted" / "archive.json"
rows = []
for path in sorted(DETAIL_DIR.glob("*/photometry.json")):
    payload = json.loads(path.read_text())["payload"]["data"]
    records = payload.get("photometry", payload.get("data", [])) if isinstance(payload, dict) else payload
    rows.extend({"source_id": path.parent.name, **record} for record in records)
photometry = pd.DataFrame(rows)
print(f"Loaded photometry rows: {len(photometry):,}")
assert len(photometry) == 7968, f"Expected 7,968 photometry rows, found {len(photometry):,}"
epoch = pd.Timestamp("1858-11-17", tz="UTC")
photometry["observation_time_utc"] = epoch + pd.to_timedelta(photometry["mjd"], unit="D")
photometry["created_at"] = pd.to_datetime(photometry["created_at"], utc=True)

Loaded photometry rows: 7,968


In [2]:
patterns = [re.compile(r"https?://gcn\.nasa\.gov/circulars/(\d+)", re.I), re.compile(r"\bGCN(?:\s+CIRCULAR)?\s*[#:]?\s*(\d{4,6})\b", re.I)]
def circular_references(value):
    text = json.dumps(value, ensure_ascii=False) if isinstance(value, dict) else ""
    return sorted({int(item) for pattern in patterns for item in pattern.findall(text)})
photometry["references"] = photometry["altdata"].map(circular_references)
origin_is_gcn = photometry["origin"].astype(str).str.strip().str.casefold().eq("gcn")
circular_ids = []
for references, is_gcn in zip(photometry["references"], origin_is_gcn):
    circular_ids.append(references[0] if is_gcn and len(references) == 1 and (ARCHIVE_DIR / f"{references[0]}.json").exists() else pd.NA)
photometry["circular_id"] = pd.array(circular_ids, dtype="Int64")
print(f"Rows matched to a circular: {photometry['circular_id'].notna().sum():,}")

Rows matched to a circular: 958


## 1. The question

A photometric measurement has three moments: when it was observed (`mjd`), when the GCN circular announcing it was published, and when it entered SkyPortal (`created_at`). This notebook measures the time between them.

## 2. One row, three moments

The GRB241030 control row shows the complete timing chain for one measurement.

In [3]:
publication_by_id = {}
for circular_id in photometry["circular_id"].dropna().unique():
    record = json.loads((ARCHIVE_DIR / f"{circular_id}.json").read_text())
    publication_by_id[int(circular_id)] = pd.to_datetime(record["createdOn"], unit="ms", utc=True)
photometry["publication_time_utc"] = pd.to_datetime(
    photometry["circular_id"].map(publication_by_id), utc=True
)
control = photometry[(photometry["source_id"] == "GRB241030") & (photometry["id"] == 37670)].iloc[0]
moments = pd.DataFrame([
    {"moment": "observation", "timestamp": control["observation_time_utc"]},
    {"moment": "GCN publication", "timestamp": control["publication_time_utc"]},
    {"moment": "SkyPortal entry", "timestamp": control["created_at"]},
])
moments["hours_since_observation"] = (
    moments["timestamp"] - control["observation_time_utc"]
).dt.total_seconds() / 3600
print(moments.round({"hours_since_observation": 3}).to_string(index=False))

         moment                           timestamp  hours_since_observation
    observation 2024-10-30 05:50:38.399999732+00:00                    0.000
GCN publication    2024-10-30 15:42:42.104000+00:00                    9.868
SkyPortal entry    2024-10-30 16:56:27.973714+00:00                   11.097


## 3. The three delays

The matched rows separate publication speed from subsequent SkyPortal ingestion.

In [4]:
photometry["lag_obs_to_publication_h"] = (
    photometry["publication_time_utc"] - photometry["observation_time_utc"]
).dt.total_seconds() / 3600
photometry["lag_publication_to_skyportal_h"] = (
    photometry["created_at"] - photometry["publication_time_utc"]
).dt.total_seconds() / 3600
photometry["lag_obs_to_skyportal_h"] = (
    photometry["created_at"] - photometry["observation_time_utc"]
).dt.total_seconds() / 3600
delays = {
    "observation to GCN publication": "lag_obs_to_publication_h",
    "GCN publication to SkyPortal": "lag_publication_to_skyportal_h",
    "observation to SkyPortal": "lag_obs_to_skyportal_h",
}
summary = []
for label, column in delays.items():
    values = photometry.loc[photometry["circular_id"].notna(), column]
    summary.append({"delay": label, "n": len(values), "median_h": values.median(), "median_d": values.median() / 24, "p90_h": values.quantile(0.9), "p90_d": values.quantile(0.9) / 24})
print(pd.DataFrame(summary).round(3).to_string(index=False))

                         delay   n  median_h  median_d    p90_h   p90_d
observation to GCN publication 958    10.732     0.447   58.269   2.428
  GCN publication to SkyPortal 958    76.695     3.196 5448.715 227.030
      observation to SkyPortal 958   100.982     4.208 5648.918 235.372


## 4. The other photometry

Rows without a matched circular retain only the observation-to-SkyPortal delay.

In [5]:
matched = photometry["circular_id"].notna()
groups = [
    ("matched to a circular", matched),
    ("not matched to a circular", ~matched),
]
comparison = []
for label, mask in groups:
    values = photometry.loc[mask, "lag_obs_to_skyportal_h"] / 24
    comparison.append({
        "group": label,
        "n": len(values),
        "median_d": values.median(),
        "p90_d": values.quantile(0.9),
    })
print(pd.DataFrame(comparison).round(3).to_string(index=False))

                    group    n  median_d   p90_d
    matched to a circular  958     4.208 235.372
not matched to a circular 7010    45.528 265.841


Matched circular measurements reach SkyPortal much sooner than unmatched photometry in this frozen capture.

In [6]:
EVIDENCE_PATH = ROOT / "notebooks" / "evidence" / "04_photometry_lag.csv"
photometry["origin_raw"] = photometry["origin"]
photometry["has_gcn_reference"] = photometry["altdata"].map(
    lambda value: isinstance(value, dict) and "gcn" in json.dumps(value, ensure_ascii=False).casefold()
)
photometry["instrument"] = photometry["instrument_name"]
photometry["band_raw"] = photometry["filter"]
evidence_columns = ["source_id", "origin_raw", "has_gcn_reference", "circular_id", "observation_time_utc", "publication_time_utc", "created_at", "lag_obs_to_publication_h", "lag_publication_to_skyportal_h", "lag_obs_to_skyportal_h", "instrument", "band_raw"]
evidence = photometry[evidence_columns].copy()
evidence.to_csv(EVIDENCE_PATH, index=False)
print(f"Evidence shape: {evidence.shape}")
print(evidence.head(3).to_string(index=False))

Evidence shape: (7968, 12)
source_id origin_raw  has_gcn_reference  circular_id                observation_time_utc publication_time_utc                       created_at  lag_obs_to_publication_h  lag_publication_to_skyportal_h  lag_obs_to_skyportal_h    instrument band_raw
 2024abfo       None              False         <NA> 2024-11-14 07:32:56.256014402+00:00                  NaT 2024-11-16 12:40:56.173255+00:00                       NaN                             NaN               53.133310         ATLAS   atlaso
 2024abfo       None              False         <NA> 2024-11-15 22:54:00.287983419+00:00                  NaT 2024-11-16 12:40:55.596289+00:00                       NaN                             NaN               13.782030         ATLAS   atlaso
 2024abfo    stdview              False         <NA> 2024-11-16 16:44:15.661999734+00:00                  NaT 2024-12-10 23:16:05.501096+00:00                       NaN                             NaN              582.530511 Les-

## 5. What this means

The GCN channel publishes measurements in hours. SkyPortal ingestion is much slower and partly retrospective, so `created_at` can date a fact substantially later than its first public availability. Temporal reconstruction must therefore preserve both publication and database-entry times. Notebook 04b contains the matching, censoring, bulk-date, and provenance checks behind these numbers.